# Stage07N — A100 gold fulltrain + private inference
Chọn **Runtime → Change runtime type → A100 GPU**. Train trên cả 6991 gold query bằng model gốc `Qwen/Qwen3-Reranker-0.6B`, không có distillation. Session L4 tiếp tục DEV/CERT độc lập. Checkpoint A100 chỉ được dùng để nộp sau khi L4 báo `PROMOTE_TO_FULLTRAIN`.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'A100' in gpu, f'Expected A100, got {gpu}'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Stage07B gold payload từ Drive
Cell này copy archive 183 MB đã dùng cho L4 từ `MyDrive/DSC2026/stage07b/`, xác minh SHA256, rồi giải nén vào `/content/stage07b/payload/`.


In [ ]:
from pathlib import Path
import hashlib, shutil, json, subprocess, os, zipfile
drive_root = Path('/content/drive/MyDrive/DSC2026')
stage = drive_root / 'stage07b'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
source = stage / 'stage07b_colab_payload.tar.gz'
assert source.is_file(), f'Missing gold payload: {source}'
assert sha256(source) == '6649f81738e1f437367bde040b1f8ac55a6d2d1fbd777998df3bd06148084eb3'
local_gold = Path('/content/stage07b_colab_payload.tar.gz')
shutil.copy2(source, local_gold)
assert sha256(local_gold) == sha256(source)
print('Gold payload copied:', local_gold, local_gold.stat().st_size)


In [ ]:
%cd /content
!tar -xzf stage07b_colab_payload.tar.gz
!ls -lh /content/stage07b/payload/MANIFEST.json /content/stage07b/payload/evidence_top20.pkl


## Copy và giải nén script bundle từ thư mục con trên Drive
Bundle nằm ở `MyDrive/DSC2026/stage07b/stage07n_a100_fulltrain/`; notebook và ba script đi cùng nhau. Fulltrain chỉ đổi training folds thành 0–4 và output folder, giữ nguyên gold loss và evidence.


In [ ]:
bundle_source = stage / 'stage07n_a100_fulltrain/stage07n_a100_fulltrain_bundle.zip'
assert bundle_source.is_file(), f'Missing script bundle: {bundle_source}'
bundle_local = Path('/content/stage07n_a100_fulltrain_bundle.zip')
shutil.copy2(bundle_source, bundle_local)
print('Script bundle copied:', bundle_local, bundle_local.stat().st_size)


In [ ]:
%cd /content
!unzip -oq stage07n_a100_fulltrain_bundle.zip -d /content
!ls -lh /content/stage07b/run_qwen06b_gold_supervised_l4.py /content/run_gold_qwen_fulltrain_a100.py /content/run_gold_qwen_private_colab.py


In [ ]:
expected_scripts = {
    '/content/stage07b/run_qwen06b_gold_supervised_l4.py': '0b15146b9f67223a99b58271f190409e9d1a2bea310b50479b24dd5da06ab339',
    '/content/run_gold_qwen_fulltrain_a100.py': 'f8b5775985c1b3ac3d38e7badbd4b187ee90c9c77935fe94baff76064509e95b',
    '/content/run_gold_qwen_private_colab.py': 'e796d2c905cd02a27e1ba77fcc430b84f06c2f23dc702d6188c2f3a3fd722713',
}
for path, expected_hash in expected_scripts.items():
    assert sha256(Path(path)) == expected_hash, f'Script hash mismatch: {path}'
print('All script hashes verified')


In [ ]:
!pip -q install 'transformers>=4.51.0,<5' accelerate safetensors scikit-learn


## Start A100 fulltrain now
L4 có thể vẫn đang train/evaluate. A100 lưu checkpoint ngay vào `MyDrive/DSC2026/stage07n_full_gold_qwen_a100/`. Cell này có thể mất khoảng 30–90 phút; thời gian thực phụ thuộc A100.


In [ ]:
train_log = drive_root / 'stage07n_full_gold_qwen_a100/FULLTRAIN_CONSOLE.log'
train_log.parent.mkdir(parents=True, exist_ok=True)
with train_log.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(['python', '-u', '/content/run_gold_qwen_fulltrain_a100.py'],
                               cwd='/content', stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line); log.flush()
    exit_code = process.wait()
assert exit_code == 0, f'Fulltrain failed, exit {exit_code}; inspect {train_log}'
print('A100 fulltrain checkpoint:', drive_root / 'stage07n_full_gold_qwen_a100/train012_model')


## Sau khi L4 DEV/CERT PASS: private inference
Chỉ chạy các cell dưới khi `MyDrive/DSC2026/stage07n_t4_gold_qwen/REPORT.json` có `decision=PROMOTE_TO_FULLTRAIN`. Private inference dùng checkpoint A100 fulltrain, output ZIP lưu về Drive.


In [ ]:
gate_path = drive_root / 'stage07n_t4_gold_qwen/REPORT.json'
assert gate_path.is_file(), 'L4 chưa xong DEV/CERT'
gate = json.loads(gate_path.read_text(encoding='utf-8'))
print('L4 decision:', gate.get('decision', gate.get('status')))
print('DEV:', gate.get('dev', {}).get('delta_recall'))
print('CERT:', (gate.get('cert') or {}).get('delta_recall'))
assert gate.get('decision') == 'PROMOTE_TO_FULLTRAIN', 'Gate failed; do not submit A100 fulltrain'
private_src = stage / 'stage07n_private_payload.tar.gz'
assert private_src.is_file(), f'Missing private evidence: {private_src}'
assert sha256(private_src) == 'c2743b544fc885f1c1287375ba45a6a9cd8ecf9e33cd2e135f007f45d8ad77b9'
private_local = Path('/content/stage07n_private_payload.tar.gz')
shutil.copy2(private_src, private_local)
assert sha256(private_local) == sha256(private_src)


In [ ]:
%cd /content
!tar -xzf stage07n_private_payload.tar.gz
!ls -lh /content/stage07n_private_payload/MANIFEST.json


In [ ]:
environment = os.environ.copy()
environment['STAGE07N_CHECKPOINT_ROOT'] = str(drive_root / 'stage07n_full_gold_qwen_a100')
infer_log = drive_root / 'stage07n_full_gold_qwen_a100/PRIVATE_CONSOLE.log'
with infer_log.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(['python', '-u', '/content/run_gold_qwen_private_colab.py'],
                               cwd='/content', env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line); log.flush()
    exit_code = process.wait()
assert exit_code == 0, f'Private inference failed, exit {exit_code}; inspect {infer_log}'


In [ ]:
out = drive_root / 'stage07n_full_gold_qwen_a100/private_inference'
zp = out / 'GOLD_QWEN06B_FULLTRAIN_PRIVATE_K5.zip'
report = json.loads((out / 'PRIVATE_SUBMISSION_REPORT.json').read_text(encoding='utf-8'))
with zipfile.ZipFile(zp) as z:
    assert z.namelist() == ['submission.json']
    rows = json.loads(z.read('submission.json'))
assert len(rows) == 2080 and all(len(v['answer']) == 5 and len(set(v['answer'])) == 5 for v in rows.values())
assert report['teacher_used'] is False and report['distillation_used'] is False
print('READY:', zp)
print('SHA256:', sha256(zp), 'queries:', len(rows))
